# Predicting Fraud in Financial Payment Services

This notebook reproduces the packaged training workflow. The labels are deterministic synthetic outcomes for demonstration; they must be replaced with approved, labeled payment data before production use.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

PACKAGE = Path.cwd()
if not (PACKAGE / 'train.py').exists():
    PACKAGE = next(p for p in [Path.cwd(), Path.cwd() / 'fraud_payment_package'] if (p / 'train.py').exists())
sys.path.insert(0, str(PACKAGE))
from train import make_dataset, pipeline, FEATURES
from sklearn.ensemble import RandomForestClassifier
data = make_dataset()
data.head()

In [ ]:
data['is_fraud'].value_counts(normalize=True).rename({0: 'legitimate', 1: 'fraud'}).plot.bar(title='Class balance')
plt.ylabel('Share')
plt.show()
data.describe(include='all').T

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
train_df, test_df = train_test_split(data, test_size=.2, random_state=42, stratify=data['is_fraud'])
model = pipeline(RandomForestClassifier(n_estimators=250, min_samples_leaf=3, class_weight='balanced', random_state=42, n_jobs=-1))
model.fit(train_df[FEATURES], train_df['is_fraud'])
pred = model.predict(test_df[FEATURES])
print(classification_report(test_df['is_fraud'], pred, target_names=['legitimate', 'fraud']))
ConfusionMatrixDisplay.from_predictions(test_df['is_fraud'], pred, display_labels=['legitimate', 'fraud'])
plt.show()

In [ ]:
from predict import FraudPaymentClassifier
classifier = FraudPaymentClassifier(PACKAGE.parents[1] / 'ml' / 'artifacts' / 'transactions')
classifier.predict({
    'amount': 4500, 'hour': 2, 'is_new_beneficiary': 1, 'is_international': 1,
    'velocity_24h': 8, 'account_age_days': 30, 'currency': 'AUD',
    'payment_channel': 'international_transfer', 'beneficiary_country': 'NG'
})

## Interpretation

Fraud recall measures how many fraudulent payments were detected; precision measures how many flagged payments were actually fraudulent. Thresholds and operational actions should be calibrated using a time-based, manually reviewed production sample, not this synthetic demonstration.